# Phase 3 — LSTM seq2seq (no attention)
We train an encoder–decoder LSTM on the 5M C4_200M GEC pairs. The encoder is a 1-layer BiLSTM (hidden 256). The decoder is a 1-layer LSTM (hidden 512) initialised from the encoder's final state. There is **no attention** — the decoder must squeeze every input token through one fixed-size vector. This is the **bottleneck baseline** that motivates Phase 4 (Bahdanau attention) and Phase 5 (Transformer).
**Inputs:**
- Tokenizer: 16k ByteLevel BPE (Phase 1)
- Splits: `train.tsv` (4.75M pairs), `val.tsv` (165k), `test.tsv` (85k)
**Targets:**
- Train 3–5 epochs with early stopping on validation GLEU
- Save best checkpoint by validation GLEU
- Estimated runtime on Kaggle P100: 4–6 hours

In [1]:
!pip install tokenizers nltk -q
import nltk
nltk.download("punkt", quiet=True)
import os, math, time, random, json
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tokenizers import ByteLevelBPETokenizer
from nltk.translate.gleu_score import corpus_gleu
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch :", torch.__version__)

Device: cuda
Torch : 2.10.0+cu128


## 1. Paths and config
All hyperparameters live in a single `CFG` dict — one source of truth, easy to dump alongside the checkpoint for reproducibility. Switching between smoke (10k) / dev (100k) / final (5M) is one field: `max_train`. Special-token IDs pinned to match the BPE tokenizer registration order: `PAD=0, UNK=1, BOS=2, EOS=3`.

In [ ]:
DATA_DIR  = "/kaggle/input/datasets/ibrahimhany202200518/gec-c4-5m-splits/data"
TOKEN_DIR = "/kaggle/input/datasets/ibrahimhany202200518/gec-bpe-tokenizer/tokenizer"
OUT_DIR   = "/kaggle/working/lstm_noattn"
os.makedirs(OUT_DIR, exist_ok=True)
CFG = {
    "vocab_size":    16000,
    "emb_dim":       256,
    "enc_hidden":    256,
    "dec_hidden":    512,
    "max_len":       64,
    "batch_size":    64,
    "lr":            1e-3,
    "weight_decay":  1e-5,
    "clip":          1.0,
    "epochs":        3,
    "tf_start":      1.0,
    "tf_end":        0.5,
    "log_every":     200,
    "ckpt_every":    2000,
    "max_train":     None,
    "max_val":       5_000,   # cap val for fast greedy decoding each epoch
    "use_attention": False,   # Phase 4 will flip this
}
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

## 2. Load tokenizer
Reload the 16k ByteLevel BPE tokenizer trained in Phase 1. Asserts pin the special-token IDs (`<pad>=0, <unk>=1, <s>=2, </s>=3`); a mismatch fails fast rather than corrupting training.

In [6]:
tokenizer = ByteLevelBPETokenizer(
    f"{TOKEN_DIR}/vocab.json",
    f"{TOKEN_DIR}/merges.txt",
)
tokenizer.add_special_tokens(["<pad>", "<unk>", "<s>", "</s>"])
assert tokenizer.token_to_id("<pad>") == PAD_ID
assert tokenizer.token_to_id("<s>")   == BOS_ID
assert tokenizer.token_to_id("</s>")  == EOS_ID
print("Vocab size:", tokenizer.get_vocab_size())

Vocab size: 16000


## 3. Dataset and collate
`GECDataset` lazy-loads `(corrupted, clean)` pairs and tokenizes on the fly. Encoder input has no special tokens; decoder target is wrapped as `[<s>, ..., </s>]` so training can use the standard shifted-right trick (feed `tgt[:-1]`, predict `tgt[1:]`). `collate_fn` does **dynamic padding** — pad each batch to the longest sequence in that batch instead of to `max_len=64`. Encoder inputs are passed through `pack_padded_sequence` so the BiLSTM skips compute over pad positions.

In [7]:
class GECDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=64, limit=None):
        self.pairs = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.rstrip("\n").split("\t")
                if len(parts) != 2:
                    continue
                self.pairs.append((parts[0], parts[1]))
                if limit is not None and len(self.pairs) >= limit:
                    break
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        src, tgt = self.pairs[i]
        src_ids = self.tok.encode(src).ids[: self.max_len]
        tgt_ids = self.tok.encode(tgt).ids[: self.max_len - 2]
        tgt_ids = [BOS_ID] + tgt_ids + [EOS_ID]
        return torch.tensor(src_ids, dtype=torch.long), \
               torch.tensor(tgt_ids, dtype=torch.long)
def collate(batch):
    srcs, tgts = zip(*batch)
    src_lens = torch.tensor([len(s) for s in srcs], dtype=torch.long)
    tgt_lens = torch.tensor([len(t) for t in tgts], dtype=torch.long)
    src_pad = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)
    tgt_pad = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)
    return src_pad, src_lens, tgt_pad, tgt_lens
train_ds = GECDataset(f"{DATA_DIR}/train.tsv", tokenizer, CFG["max_len"], CFG["max_train"])
val_ds   = GECDataset(f"{DATA_DIR}/val.tsv",   tokenizer, CFG["max_len"], CFG["max_val"])
test_ds  = GECDataset(f"{DATA_DIR}/test.tsv",  tokenizer, CFG["max_len"])
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=2, pin_memory=True, collate_fn=collate)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=2, pin_memory=True, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=2, pin_memory=True, collate_fn=collate)
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")

Train: 4,750,000  Val: 5,000  Test: 85,000


## 4. Model
**Encoder.** Embedding(16k, 256) → 1-layer BiLSTM(hidden=256). Final forward and backward `(h, c)` are concatenated and projected (Linear + tanh) to size `dec_hidden=512` to serve as the decoder's initial state.
**Decoder.** Embedding(16k, 256) → 1-layer LSTM(hidden=512) → Linear(512 → 16k). Initial state comes from the encoder bottleneck. During training the whole shifted-right target runs through the decoder in parallel (teacher forcing).
**Seq2Seq wrapper** carries a `use_attention` flag. In Phase 3 it is `False` (decoder only sees the bottleneck vector). Phase 4 will flip it `True` and add a Bahdanau attention module that lets the decoder query the *full sequence* of encoder states.
`greedy_decode` runs the decoder one step at a time at inference, stopping when every beam has emitted `</s>` or `max_len` is reached.

In [8]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, pad_id=PAD_ID):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(emb_dim, enc_hidden, batch_first=True, bidirectional=True)
        self.h_proj = nn.Linear(enc_hidden * 2, dec_hidden)
        self.c_proj = nn.Linear(enc_hidden * 2, dec_hidden)
    def forward(self, src, src_lens):
        emb = self.embedding(src)
        packed = nn.utils.rnn.pack_padded_sequence(emb, src_lens.cpu(),
                                                   batch_first=True, enforce_sorted=False)
        outputs, (h, c) = self.lstm(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)
        # h, c: (2, B, enc_hidden) → concat last fwd+bwd → project
        h_cat = torch.cat([h[0], h[1]], dim=-1)
        c_cat = torch.cat([c[0], c[1]], dim=-1)
        h0 = torch.tanh(self.h_proj(h_cat)).unsqueeze(0)   # (1, B, dec_hidden)
        c0 = torch.tanh(self.c_proj(c_cat)).unsqueeze(0)
        return outputs, (h0, c0)
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, dec_hidden, pad_id=PAD_ID):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(emb_dim, dec_hidden, batch_first=True)
        self.out = nn.Linear(dec_hidden, vocab_size)
    def forward(self, tgt_in, state):
        emb = self.embedding(tgt_in)
        out, state = self.lstm(emb, state)
        logits = self.out(out)
        return logits, state
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, use_attention=False):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.use_attention = use_attention   # Phase 4 hook
    def forward(self, src, src_lens, tgt_in):
        _, state = self.encoder(src, src_lens)
        logits, _ = self.decoder(tgt_in, state)
        return logits
    @torch.no_grad()
    def greedy_decode(self, src, src_lens, max_len=64):
        self.eval()
        B = src.size(0)
        _, state = self.encoder(src, src_lens)
        ys = torch.full((B, 1), BOS_ID, dtype=torch.long, device=src.device)
        finished = torch.zeros(B, dtype=torch.bool, device=src.device)
        outputs = []
        for _ in range(max_len):
            logits, state = self.decoder(ys[:, -1:], state)
            next_tok = logits[:, -1].argmax(-1)
            next_tok = torch.where(finished, torch.full_like(next_tok, PAD_ID), next_tok)
            outputs.append(next_tok)
            ys = torch.cat([ys, next_tok.unsqueeze(1)], dim=1)
            finished = finished | (next_tok == EOS_ID)
            if finished.all():
                break
        return torch.stack(outputs, dim=1)
encoder = Encoder(CFG["vocab_size"], CFG["emb_dim"], CFG["enc_hidden"], CFG["dec_hidden"])
decoder = Decoder(CFG["vocab_size"], CFG["emb_dim"], CFG["dec_hidden"])
model   = Seq2Seq(encoder, decoder, use_attention=CFG["use_attention"]).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params/1e6:.2f}M")

Trainable params: 19.55M


## 5. Loss, optimizer, scheduler
- **Loss:** `CrossEntropyLoss(ignore_index=PAD_ID)` — padding positions contribute nothing.
- **Optimizer:** Adam, `lr=1e-3`, `weight_decay=1e-5`.
- **Scheduler:** `ReduceLROnPlateau(factor=0.5, patience=2)` on validation loss.
- **Mixed precision** via `torch.cuda.amp` with `GradScaler` — ~2× throughput, half the VRAM.
- **Gradient clipping** at global L2 norm 1.0 — prevents LSTM gradient explosion.
- **Teacher forcing** ratio decays linearly 1.0 → 0.5 across epochs.

In [12]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(),
                             lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2)
scaler = GradScaler()

/tmp/ipykernel_57/3333420250.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


## 6. Validation: greedy decode + GLEU
Each epoch we compute two numbers on the validation split: teacher-forced loss (for the LR scheduler) and corpus GLEU (for early stopping / best-checkpoint selection). Greedy decoding is used here for speed; beam search is reserved for the final test pass. Validation is capped to `max_val=5000` pairs so each epoch's eval takes minutes, not hours.

In [13]:
def ids_to_text(ids):
    out = []
    for i in ids:
        i = int(i)
        if i in (BOS_ID, PAD_ID):
            continue
        if i == EOS_ID:
            break
        out.append(i)
    return tokenizer.decode(out)
@torch.no_grad()
def evaluate_gleu(model, loader, max_batches=None):
    model.eval()
    refs, hyps = [], []
    total_loss, total_tokens = 0.0, 0
    for bi, (src, src_lens, tgt, tgt_lens) in enumerate(loader):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        src_lens = src_lens.to(DEVICE)
        # teacher-forced loss for monitoring
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        with autocast():
            logits = model(src, src_lens, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        n_tok = (tgt_out != PAD_ID).sum().item()
        total_loss   += loss.item() * n_tok
        total_tokens += n_tok
        # greedy decode for GLEU
        gen = model.greedy_decode(src, src_lens, max_len=CFG["max_len"])
        for g, t in zip(gen.cpu().tolist(), tgt.cpu().tolist()):
            hyps.append(ids_to_text(g).split())
            refs.append([ids_to_text(t).split()])
        if max_batches is not None and bi + 1 >= max_batches:
            break
    avg_loss = total_loss / max(total_tokens, 1)
    gleu = corpus_gleu(refs, hyps)
    return avg_loss, gleu

## 7. Training loop
Standard mixed-precision loop: forward → CE loss with padding ignored → scaled backward → unscale and clip gradients → optimizer step → scaler update. Loss is tracked per non-pad token (more meaningful than per-batch since batches have different effective lengths). `last.pt` is checkpointed every `ckpt_every` steps so a Kaggle session timeout costs minutes, not hours. `best.pt` is updated only when validation GLEU improves.

In [14]:
def save_ckpt(path, model, optimizer, step, epoch, best_gleu):
    torch.save({
        "model":     model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step":      step,
        "epoch":     epoch,
        "best_gleu": best_gleu,
        "cfg":       CFG,
    }, path)
def train_one_epoch(model, loader, epoch, global_step, best_gleu, tf_ratio):
    model.train()
    running_loss, running_tokens = 0.0, 0
    t0 = time.time()
    for src, src_lens, tgt, tgt_lens in loader:
        src, tgt = src.to(DEVICE, non_blocking=True), tgt.to(DEVICE, non_blocking=True)
        src_lens = src_lens.to(DEVICE)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(src, src_lens, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["clip"])
        scaler.step(optimizer)
        scaler.update()
        n_tok = (tgt_out != PAD_ID).sum().item()
        running_loss   += loss.item() * n_tok
        running_tokens += n_tok
        global_step += 1
        if global_step % CFG["log_every"] == 0:
            cur = running_loss / max(running_tokens, 1)
            lr = optimizer.param_groups[0]["lr"]
            dt = time.time() - t0
            print(f"  step {global_step:>7} | loss {cur:.4f} | ppl {math.exp(cur):.1f} "
                  f"| lr {lr:.2e} | tf {tf_ratio:.2f} | {dt:.0f}s")
            running_loss, running_tokens, t0 = 0.0, 0, time.time()
        if global_step % CFG["ckpt_every"] == 0:
            save_ckpt(f"{OUT_DIR}/last.pt", model, optimizer, global_step, epoch, best_gleu)
    return global_step

In [15]:
best_gleu = -1.0
global_step = 0
for epoch in range(1, CFG["epochs"] + 1):
    frac = (epoch - 1) / max(CFG["epochs"] - 1, 1)
    tf_ratio = CFG["tf_start"] + frac * (CFG["tf_end"] - CFG["tf_start"])
    print(f"\n=== Epoch {epoch}/{CFG['epochs']} (teacher-forcing={tf_ratio:.2f}) ===")
    global_step = train_one_epoch(model, train_loader, epoch, global_step, best_gleu, tf_ratio)
    print("Validating...")
    val_loss, val_gleu = evaluate_gleu(model, val_loader)
    print(f"epoch {epoch}: val_loss={val_loss:.4f}  val_gleu={val_gleu:.4f}")
    scheduler.step(val_loss)
    save_ckpt(f"{OUT_DIR}/last.pt", model, optimizer, global_step, epoch, best_gleu)
    if val_gleu > best_gleu:
        best_gleu = val_gleu
        save_ckpt(f"{OUT_DIR}/best.pt", model, optimizer, global_step, epoch, best_gleu)
        print(f"  ↳ new best, saved to {OUT_DIR}/best.pt")


=== Epoch 1/3 (teacher-forcing=1.00) ===


/tmp/ipykernel_57/730971936.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  step     200 | loss 7.3448 | ppl 1548.1 | lr 1.00e-03 | tf 1.00 | 9s
  step     400 | loss 6.6871 | ppl 802.0 | lr 1.00e-03 | tf 1.00 | 7s
  step     600 | loss 6.3439 | ppl 569.0 | lr 1.00e-03 | tf 1.00 | 7s
  step     800 | loss 6.1119 | ppl 451.2 | lr 1.00e-03 | tf 1.00 | 7s
  step    1000 | loss 5.9113 | ppl 369.2 | lr 1.00e-03 | tf 1.00 | 7s
  step    1200 | loss 5.7540 | ppl 315.4 | lr 1.00e-03 | tf 1.00 | 7s
  step    1400 | loss 5.6066 | ppl 272.2 | lr 1.00e-03 | tf 1.00 | 8s
  step    1600 | loss 5.4830 | ppl 240.6 | lr 1.00e-03 | tf 1.00 | 8s
  step    1800 | loss 5.3799 | ppl 217.0 | lr 1.00e-03 | tf 1.00 | 8s
  step    2000 | loss 5.2593 | ppl 192.3 | lr 1.00e-03 | tf 1.00 | 8s
  step    2200 | loss 5.1747 | ppl 176.7 | lr 1.00e-03 | tf 1.00 | 8s
  step    2400 | loss 5.0584 | ppl 157.3 | lr 1.00e-03 | tf 1.00 | 8s
  step    2600 | loss 4.9957 | ppl 147.8 | lr 1.00e-03 | tf 1.00 | 8s
  step    2800 | loss 4.9039 | ppl 134.8 | lr 1.00e-03 | tf 1.00 | 8s
  step    3000 | lo

/tmp/ipykernel_57/2228293680.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


epoch 1: val_loss=2.6013  val_gleu=0.1905
  ↳ new best, saved to /kaggle/working/lstm_noattn/best.pt

=== Epoch 2/3 (teacher-forcing=0.75) ===
  step   74400 | loss 2.5837 | ppl 13.2 | lr 1.00e-03 | tf 0.75 | 8s
  step   74600 | loss 2.5673 | ppl 13.0 | lr 1.00e-03 | tf 0.75 | 8s
  step   74800 | loss 2.5713 | ppl 13.1 | lr 1.00e-03 | tf 0.75 | 8s
  step   75000 | loss 2.5954 | ppl 13.4 | lr 1.00e-03 | tf 0.75 | 8s
  step   75200 | loss 2.5703 | ppl 13.1 | lr 1.00e-03 | tf 0.75 | 8s
  step   75400 | loss 2.5599 | ppl 12.9 | lr 1.00e-03 | tf 0.75 | 8s
  step   75600 | loss 2.5614 | ppl 13.0 | lr 1.00e-03 | tf 0.75 | 8s
  step   75800 | loss 2.5813 | ppl 13.2 | lr 1.00e-03 | tf 0.75 | 8s
  step   76000 | loss 2.5744 | ppl 13.1 | lr 1.00e-03 | tf 0.75 | 8s
  step   76200 | loss 2.5693 | ppl 13.1 | lr 1.00e-03 | tf 0.75 | 8s
  step   76400 | loss 2.5862 | ppl 13.3 | lr 1.00e-03 | tf 0.75 | 8s
  step   76600 | loss 2.5884 | ppl 13.3 | lr 1.00e-03 | tf 0.75 | 8s
  step   76800 | loss 2.5696 

## 8. Test set: greedy + beam=4
Load `best.pt` and decode the full test set twice. **Greedy** (pick argmax each step) is fast and gives the speed-quality reference. **Beam=4** with length-normalised scoring (α=0.7) is the final reported result. Both prediction files are saved as `(src, pred, ref)` TSV so they can be re-scored later without re-running decoding.

In [ ]:
ckpt = torch.load(f"{OUT_DIR}/best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
print("Loaded best checkpoint from epoch", ckpt["epoch"], "step", ckpt["step"])
preds_greedy, refs_text, srcs_text = [], [], []
model.eval()
with torch.no_grad():
    for src, src_lens, tgt, tgt_lens in test_loader:
        src = src.to(DEVICE); src_lens = src_lens.to(DEVICE)
        gen = model.greedy_decode(src, src_lens, max_len=CFG["max_len"])
        for s, g, t in zip(src.cpu().tolist(), gen.cpu().tolist(), tgt.cpu().tolist()):
            srcs_text.append(ids_to_text(s))
            preds_greedy.append(ids_to_text(g))
            refs_text.append(ids_to_text(t))
with open(f"{OUT_DIR}/test_predictions_greedy.tsv", "w", encoding="utf-8") as f:
    for s, p, r in zip(srcs_text, preds_greedy, refs_text):
        f.write(f"{s}\t{p}\t{r}\n")
print("Saved greedy predictions:", f"{OUT_DIR}/test_predictions_greedy.tsv")

Loaded best checkpoint from epoch 3 step 222657
Saved greedy predictions: /kaggle/working/lstm_noattn/test_predictions_greedy.tsv


In [17]:
@torch.no_grad()
def beam_decode_one(model, src_ids, src_len, beam=4, max_len=64, alpha=0.7):
    model.eval()
    src = src_ids.unsqueeze(0).to(DEVICE)
    src_lens = torch.tensor([src_len], device=DEVICE)
    _, state = model.encoder(src, src_lens)
    h, c = state
    # initial beam: (score, tokens, h, c, finished)
    beams = [(0.0, [BOS_ID], h, c, False)]
    for _ in range(max_len):
        new_beams = []
        for score, toks, hh, cc, fin in beams:
            if fin:
                new_beams.append((score, toks, hh, cc, fin)); continue
            last = torch.tensor([[toks[-1]]], device=DEVICE)
            logits, (hh2, cc2) = model.decoder(last, (hh, cc))
            log_probs = F.log_softmax(logits[:, -1], dim=-1).squeeze(0)
            top_lp, top_ix = log_probs.topk(beam)
            for lp, ix in zip(top_lp.tolist(), top_ix.tolist()):
                new_toks = toks + [ix]
                new_fin  = (ix == EOS_ID)
                new_beams.append((score + lp, new_toks, hh2, cc2, new_fin))
        # length-normalized score
        new_beams.sort(key=lambda b: b[0] / (len(b[1]) ** alpha), reverse=True)
        beams = new_beams[:beam]
        if all(b[4] for b in beams):
            break
    best = beams[0][1]
    return best
# Beam=4 on full test (expect ~30–60 min on P100; reduce test slice if needed)
preds_beam = []
for i, (src, tgt) in enumerate(test_ds):
    out_ids = beam_decode_one(model, src, len(src), beam=4)
    preds_beam.append(ids_to_text(out_ids))
    if (i + 1) % 5000 == 0:
        print(f"  beam decoded {i+1}/{len(test_ds)}")
with open(f"{OUT_DIR}/test_predictions_beam4.tsv", "w", encoding="utf-8") as f:
    for s, p, r in zip(srcs_text, preds_beam, refs_text):
        f.write(f"{s}\t{p}\t{r}\n")
        

  beam decoded 5000/85000
  beam decoded 10000/85000
  beam decoded 15000/85000
  beam decoded 20000/85000
  beam decoded 25000/85000
  beam decoded 30000/85000
  beam decoded 35000/85000
  beam decoded 40000/85000
  beam decoded 45000/85000
  beam decoded 50000/85000
  beam decoded 55000/85000
  beam decoded 60000/85000
  beam decoded 65000/85000
  beam decoded 70000/85000
  beam decoded 75000/85000
  beam decoded 80000/85000
  beam decoded 85000/85000


## 9. Metrics
Four numbers per decoding strategy:
| Metric | What it measures |
|--------|-----------------|
| **Exact-match** | output equals reference token-for-token |
| **GLEU** | n-gram overlap with reference, penalising unfixed source overlap |
| **Precision / Recall / F0.5** | token-level retrieval framing; β=0.5 weights precision higher (GEC convention) |
| **Length-bucketed accuracy** | exact-match split by reference length: short ≤10, medium 11–20, long >20 |

In [18]:
def exact_match(preds, refs):
    return sum(p.strip() == r.strip() for p, r in zip(preds, refs)) / len(preds)
def token_prf(preds, refs, beta=0.5):
    tp = fp = fn = 0
    for p, r in zip(preds, refs):
        ps, rs = p.split(), r.split()
        from collections import Counter
        pc, rc = Counter(ps), Counter(rs)
        for tok, n in pc.items():
            tp += min(n, rc.get(tok, 0))
            fp += max(0, n - rc.get(tok, 0))
        for tok, n in rc.items():
            fn += max(0, n - pc.get(tok, 0))
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f = (1 + beta**2) * prec * rec / (beta**2 * prec + rec + 1e-9)
    return prec, rec, f
def length_bucketed_em(srcs, preds, refs):
    buckets = {"short(<=10)": [], "med(11-20)": [], "long(>20)": []}
    for s, p, r in zip(srcs, preds, refs):
        n = len(r.split())
        key = "short(<=10)" if n <= 10 else "med(11-20)" if n <= 20 else "long(>20)"
        buckets[key].append(p.strip() == r.strip())
    return {k: (sum(v)/len(v) if v else 0.0, len(v)) for k, v in buckets.items()}
for name, preds in [("greedy", preds_greedy), ("beam=4", preds_beam)]:
    em = exact_match(preds, refs_text)
    refs_tok = [[r.split()] for r in refs_text]
    hyps_tok = [p.split() for p in preds]
    gleu = corpus_gleu(refs_tok, hyps_tok)
    p, r, f05 = token_prf(preds, refs_text, beta=0.5)
    buckets = length_bucketed_em(srcs_text, preds, refs_text)
    print(f"\n--- {name} ---")
    print(f"Exact-match : {em:.4f}")
    print(f"GLEU        : {gleu:.4f}")
    print(f"Precision   : {p:.4f}")
    print(f"Recall      : {r:.4f}")
    print(f"F0.5        : {f05:.4f}")
    for k, (acc, n) in buckets.items():
        print(f"  EM {k:>14}  {acc:.4f}  (n={n})")


--- greedy ---
Exact-match : 0.0080
GLEU        : 0.2128
Precision   : 0.4655
Recall      : 0.4629
F0.5        : 0.4650
  EM    short(<=10)  0.0265  (n=21199)
  EM     med(11-20)  0.0042  (n=28830)
  EM      long(>20)  0.0000  (n=34971)

--- beam=4 ---
Exact-match : 0.0118
GLEU        : 0.2560
Precision   : 0.5131
Recall      : 0.5026
F0.5        : 0.5109
  EM    short(<=10)  0.0367  (n=21199)
  EM     med(11-20)  0.0077  (n=28830)
  EM      long(>20)  0.0001  (n=34971)


## 10. Qualitative examples
Random sample of 30 test cases, with the first 10 printed in full (src / pred / ref) for hand-picking. The selected examples illustrate model behaviour across success, failure, and boundary cases for the report's linguistic-analysis section.

In [19]:
import random
random.seed(0)
indices = random.sample(range(len(srcs_text)), 30)
for i in indices[:10]:
    print(f"[{i}] SRC : {srcs_text[i]}")
    print(f"     PRED: {preds_greedy[i]}")
    print(f"     REF : {refs_text[i]}")
    print(f"     match={'✓' if preds_greedy[i].strip()==refs_text[i].strip() else '✗'}")
    print()

[50494] SRC : Apart from that 1 = csipsimple SIP for Android devices.explains how to build the method application.
     PRED: Apart from that 1s. Cucips™s for SIP apps.explains how to build the method application.
     REF : 1 csipsimple SIP application for Android devices explains how to build the application.
     match=✗

[55125] SRC : This is a work memory.
     PRED: This is a work memory.
     REF : This is a modern block.
     match=✗

[5306] SRC : The bank says statement that more hikes will be needed, but around it omitted the word “gradual” from its explanation on how it will approach future rate increases _ something that could lead to observers to anticipate future increases will come faster than as they had previously expected.
     PRED: The bank says that there would be no more than stop, but I would need the “Arganda” of its application process to have its approach to how much better return is an opportunity to change that will increase in a global terosome expected to 

## 11. Save run summary
Headline numbers (model name, parameter count, full config, best val GLEU, test metrics) saved to `summary.json` next to the checkpoint. Phase 4 will load this file and produce a side-by-side comparison plot automatically.

In [20]:
summary = {
    "model": "lstm_seq2seq_no_attention",
    "params_M": round(n_params/1e6, 2),
    "config": CFG,
    "best_val_gleu": best_gleu,
    "test": {
        "greedy": {
            "exact_match": exact_match(preds_greedy, refs_text),
            "gleu": corpus_gleu([[r.split()] for r in refs_text],
                                [p.split() for p in preds_greedy]),
        },
    },
}
with open(f"{OUT_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

{
  "model": "lstm_seq2seq_no_attention",
  "params_M": 19.55,
  "config": {
    "vocab_size": 16000,
    "emb_dim": 256,
    "enc_hidden": 256,
    "dec_hidden": 512,
    "max_len": 64,
    "batch_size": 64,
    "lr": 0.001,
    "weight_decay": 1e-05,
    "clip": 1.0,
    "epochs": 3,
    "tf_start": 1.0,
    "tf_end": 0.5,
    "log_every": 200,
    "ckpt_every": 2000,
    "max_train": null,
    "max_val": 5000,
    "use_attention": false
  },
  "best_val_gleu": 0.21210116533509746,
  "test": {
    "greedy": {
      "exact_match": 0.008035294117647059,
      "gleu": 0.21280164981902633
    }
  }
}
